In [54]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

root = "C:/Users/nefel/Desktop/Virality-on-Shorts"

youtube = pd.read_csv(root + "/data/clean/youtube_clean.csv")
instagram = pd.read_csv(root + "/data/clean/instagram_clean.csv")
tiktok = pd.read_csv(root + "/data/clean/tiktok_clean.csv")

In [55]:
print("YouTube columns:")
print(youtube.columns.tolist())

print("\nInstagram columns:")
print(instagram.columns.tolist())

print("\nTikTok columns:")
print(tiktok.columns.tolist())

YouTube columns:
['Unnamed: 0', 'domain_searched', 'publisher', 'channel_id', 'video_id', 'url', 'title', 'published_at', 'views', 'likes', 'comments', 'duration', 'time_window_start', 'time_window_end', 'description', 'year', 'engagement_rate', 'like_rate', 'comment_rate']

Instagram columns:
['Unnamed: 0', 'content_type', 'published_at', 'hashtags', 'video_id', 'is_branded_content', 'lang', 'mcl_url', 'modified_time', 'post_owner.id', 'publisher', 'post_owner.type', 'username', 'comments', 'likes', 'views', 'statistics.views_date_last_refreshed', 'description', 'duration', 'url', 'media_type', 'year_month', 'engagement_rate', 'like_rate', 'comment_rate']

TikTok columns:
['Unnamed: 0', 'video_id', 'url', 'publisher', 'description', 'hashtags', 'created_at_unix', 'published_at', 'duration', 'region_code', 'voice_to_text', 'views', 'likes', 'comments', 'shares', 'engagement_rate_%', 'music_id', 'is_stem_verified', 'scraped_at', 'year', 'year_month', 'engagement_rate', 'like_rate', 'com

In [56]:
def standardize_platform(df, platform):
    df = df.copy()
    
    df["platform"] = platform
    
    # Standardize datetime
    df["published_at"] = pd.to_datetime(
        df["published_at"],
        utc=True,
        errors="coerce"
    )
    
    df["date"] = df["published_at"].dt.date
    
    # Standardize views
    if "views" in df.columns:
        df["views"] = pd.to_numeric(df["views"], errors="coerce")
    
    return df

In [57]:
youtube = standardize_platform(youtube, "YouTube")
instagram = standardize_platform(instagram, "Instagram")
tiktok = standardize_platform(tiktok, "TikTok")

## Create a comparable text field

In [59]:
def make_text_field(df):
    df = df.copy()
    
    possible_text_cols = [
        "title",
        "description",
        "text",
        "caption",
        "hashtags",
        "voice_to_text"
    ]
    
    existing_text_cols = [
        col for col in possible_text_cols
        if col in df.columns
    ]
    
    print("Using text columns:", existing_text_cols)
    
    if len(existing_text_cols) == 0:
        df["match_text"] = ""
    else:
        df["match_text"] = (
            df[existing_text_cols]
            .fillna("")
            .astype(str)
            .agg(" ".join, axis=1)
        )
    
    return df

In [60]:
youtube = make_text_field(youtube)
instagram = make_text_field(instagram)
tiktok = make_text_field(tiktok)

Using text columns: ['title', 'description']
Using text columns: ['description', 'hashtags']
Using text columns: ['description', 'hashtags', 'voice_to_text']


## Clean text

In [61]:
def clean_text(text):
    text = str(text).lower()
    
    # remove urls
    text = re.sub(r"http\S+|www\S+", " ", text)
    
    # remove mentions
    text = re.sub(r"@\w+", " ", text)
    
    # keep hashtags as words, remove the symbol
    text = text.replace("#", " ")
    
    # remove punctuation/numbers if needed
    text = re.sub(r"[^a-zA-ZÀ-ÿ\s]", " ", text)
    
    # normalize spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [62]:
for df in [youtube, instagram, tiktok]:
    df["match_text_clean"] = df["match_text"].apply(clean_text)
    df["text_length"] = df["match_text_clean"].str.len()

## Possible account fields

In [63]:
import re

def clean_string(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def extract_tiktok_username(url):
    """
    Extract TikTok username from URLs like:
    https://www.tiktok.com/@username/video/...
    """
    if pd.isna(url):
        return ""

    url = str(url)

    match = re.search(r"tiktok\.com/@([^/?]+)", url)
    if match:
        return match.group(1).strip()

    return ""


def normalize_source_name(name):
    """
    Conservative normalization for publisher/source names.

    Goal:
    - remove formatting differences
    - keep meaningful media-name words such as news, tv, italia, online
    - avoid collapsing different outlets into generic names
    """
    name = clean_string(name).lower()

    # remove urls prefixes
    name = re.sub(r"http\S+|www\S+", " ", name)

    # remove leading @ from usernames, but keep the username text
    name = name.replace("@", " ")

    # split common domain punctuation
    name = re.sub(r"[._\-]", " ", name)

    # remove standalone domain suffixes only
    name = re.sub(r"\b(it|com|eu|net|org)\b", " ", name)

    # remove remaining punctuation
    name = re.sub(r"[^a-zA-ZÀ-ÿ0-9\s]", " ", name)

    # only remove very safe account-noise words
    noise_words = [
        "official",
        "ufficiale",
        "canale",
        "channel"
    ]

    tokens = name.split()
    tokens = [t for t in tokens if t not in noise_words]

    name = " ".join(tokens)
    name = re.sub(r"\s+", " ", name).strip()

    return name


# -----------------------
# YouTube source fields
# -----------------------

youtube["source_platform_id"] = youtube["channel_id"].apply(clean_string)
youtube["source_account_raw"] = youtube["publisher"].apply(clean_string)
youtube["source_domain_raw"] = youtube["domain_searched"].apply(clean_string)

youtube["source_identity_raw"] = (
    youtube["domain_searched"].fillna("").astype(str) + " " +
    youtube["publisher"].fillna("").astype(str)
)


# -----------------------
# Instagram source fields
# -----------------------

instagram["source_platform_id"] = instagram["post_owner.id"].apply(clean_string)
instagram["source_account_raw"] = instagram["username"].apply(clean_string)
instagram["source_domain_raw"] = instagram["publisher"].apply(clean_string)

instagram["source_identity_raw"] = (
    instagram["publisher"].fillna("").astype(str) + " " +
    instagram["username"].fillna("").astype(str)
)


# -----------------------
# TikTok source fields
# -----------------------

tiktok["source_account_raw"] = tiktok["url"].apply(extract_tiktok_username)
tiktok["source_platform_id"] = tiktok["source_account_raw"]
tiktok["source_domain_raw"] = tiktok["publisher"].apply(clean_string)

tiktok["source_identity_raw"] = (
    tiktok["publisher"].fillna("").astype(str) + " " +
    tiktok["source_account_raw"].fillna("").astype(str)
)


# -----------------------
# Normalize source fields
# -----------------------

for df in [youtube, instagram, tiktok]:

    df["source_account_norm"] = df["source_account_raw"].apply(
        normalize_source_name
    )

    df["source_domain_norm"] = df["source_domain_raw"].apply(
        normalize_source_name
    )

    df["source_identity_norm"] = df["source_identity_raw"].apply(
        normalize_source_name
    )

## Create one combined dataset

In [64]:
# =============================================================================
# CREATE COMBINED MATCHING DATASET
# =============================================================================

common_cols = [
    "platform",
    "publisher",
    "published_at",
    "date",
    "views",
    "likes",
    "comments",
    "duration",
    "match_text_clean",
    "text_length",
    "source_platform_id",
    "source_account_raw",
    "source_domain_raw",
    "source_identity_raw",
    "source_account_norm",
    "source_domain_norm",
    "source_identity_norm"
]

id_cols = ["video_id", "url"]


def keep_common_columns(df):
    df = df.copy()

    cols = []

    for col in id_cols + common_cols:
        if col in df.columns:
            cols.append(col)

    return df[cols].copy()


all_videos = pd.concat(
    [
        keep_common_columns(youtube),
        keep_common_columns(instagram),
        keep_common_columns(tiktok)
    ],
    ignore_index=True
)

all_videos = all_videos.dropna(
    subset=["publisher", "published_at", "match_text_clean"]
)

all_videos = all_videos[
    all_videos["match_text_clean"].str.len() > 20
].copy()

all_videos["published_at"] = pd.to_datetime(
    all_videos["published_at"],
    utc=True,
    errors="coerce"
)

all_videos["date"] = all_videos["published_at"].dt.date

display(all_videos.head())
print(all_videos["platform"].value_counts())

,video_id,url,platform,publisher,published_at,date,views,likes,comments,duration,match_text_clean,text_length,source_platform_id,source_account_raw,source_domain_raw,source_identity_raw,source_account_norm,source_domain_norm,source_identity_norm
0,0XnpMrwKk6A,https://www.youtube.com/shorts/0XnpMrwKk6A,YouTube,RaiNews,2026-05-31 10:04:52+00:00,2026-05-31,47466.0,304.0,484.0,PT1M15S,scontri tra tifosi e polizia a parigi dopo la ...,545,UCxqR9g_1XlnfrqwHK9viwCw,RaiNews,rainews.it,rainews.it RaiNews,rainews,rainews,rainews rainews
1,BW8JSuCP1Co,https://www.youtube.com/shorts/BW8JSuCP1Co,YouTube,RaiNews,2026-05-31 09:58:56+00:00,2026-05-31,34155.0,438.0,34.0,PT1M36S,vasco rossi il racconto del concerto vero rito...,256,UCxqR9g_1XlnfrqwHK9viwCw,RaiNews,rainews.it,rainews.it RaiNews,rainews,rainews,rainews rainews
2,0yH8exBeook,https://www.youtube.com/shorts/0yH8exBeook,YouTube,RaiNews,2026-05-31 09:07:37+00:00,2026-05-31,67727.0,703.0,299.0,PT1M20S,beatrice morta a anni il compagno della madre ...,370,UCxqR9g_1XlnfrqwHK9viwCw,RaiNews,rainews.it,rainews.it RaiNews,rainews,rainews,rainews rainews
3,V1ZxqKZMNbk,https://www.youtube.com/shorts/V1ZxqKZMNbk,YouTube,RaiNews,2026-05-30 18:33:17+00:00,2026-05-30,21721.0,330.0,57.0,PT1M28S,si è avventato contro di me sono riuscito a bl...,388,UCxqR9g_1XlnfrqwHK9viwCw,RaiNews,rainews.it,rainews.it RaiNews,rainews,rainews,rainews rainews
4,Nky2RIRPlnA,https://www.youtube.com/shorts/Nky2RIRPlnA,YouTube,RaiNews,2026-05-30 18:24:23+00:00,2026-05-30,66585.0,605.0,81.0,PT1M16S,femminicidio a ferrara mi aveva promesso che l...,419,UCxqR9g_1XlnfrqwHK9viwCw,RaiNews,rainews.it,rainews.it RaiNews,rainews,rainews,rainews rainews


platform
Instagram    230227
TikTok       192788
YouTube       51176
Name: count, dtype: int64


# Generate candidate cross-platform matches

In [65]:
from rapidfuzz import fuzz

## Check publisher overlap across platforms

In [68]:
# =============================================================================
# SOURCE OVERLAP ACROSS PLATFORMS
# =============================================================================

source_cols = [
    "platform",
    "publisher",
    "source_account_raw",
    "source_domain_raw",
    "source_account_norm",
    "source_domain_norm",
    "source_identity_norm"
]

all_sources = (
    all_videos[source_cols]
    .drop_duplicates()
    .copy()
)

for col in [
    "publisher",
    "source_domain_norm",
    "source_account_norm",
    "source_identity_norm"
]:

    overlap = (
        all_sources
        .groupby(col)["platform"]
        .agg(
            n_platforms="nunique",
            platforms=lambda x: ", ".join(sorted(x.unique()))
        )
        .reset_index()
    )

    overlap = overlap[
        (overlap[col] != "") &
        (overlap["n_platforms"] >= 2)
    ]

    print("\n" + "=" * 70)
    print(f"Overlap using: {col}")
    print("=" * 70)
    print("Sources appearing on at least 2 platforms:", len(overlap))
    print("Sources appearing on all 3 platforms:", (overlap["n_platforms"] == 3).sum())

    display(
        overlap
        .sort_values(["n_platforms", col], ascending=[False, True])
        .head(30)
    )


Overlap using: publisher
Sources appearing on at least 2 platforms: 27
Sources appearing on all 3 platforms: 0


,publisher,n_platforms,platforms
6,Adnkronos,2,"Instagram, YouTube"
16,CICAP,2,"Instagram, YouTube"
25,Cronache della Campania,2,"Instagram, YouTube"
42,Facta,2,"Instagram, YouTube"
44,Fanpage.it,2,"Instagram, YouTube"
46,Focus,2,"Instagram, YouTube"
47,Focus Junior,2,"Instagram, YouTube"
65,Geopop,2,"Instagram, YouTube"
67,Giornale La Voce,2,"Instagram, YouTube"
89,Il Tirreno,2,"Instagram, YouTube"



Overlap using: source_domain_norm
Sources appearing on at least 2 platforms: 67
Sources appearing on all 3 platforms: 19


,source_domain_norm,n_platforms,platforms
16,avvenire,3,"Instagram, TikTok, YouTube"
31,cicap,3,"Instagram, TikTok, YouTube"
70,fanpage,3,"Instagram, TikTok, YouTube"
72,focus,3,"Instagram, TikTok, YouTube"
82,gay,3,"Instagram, TikTok, YouTube"
93,geopop,3,"Instagram, TikTok, YouTube"
105,greenme,3,"Instagram, TikTok, YouTube"
133,ilcapoluogo,3,"Instagram, TikTok, YouTube"
143,ilmeteo,3,"Instagram, TikTok, YouTube"
156,internapoli,3,"Instagram, TikTok, YouTube"



Overlap using: source_account_norm
Sources appearing on at least 2 platforms: 89
Sources appearing on all 3 platforms: 13


,source_account_norm,n_platforms,platforms
30,cicap,3,"Instagram, TikTok, YouTube"
42,cronachemaceratesi,3,"Instagram, TikTok, YouTube"
70,fanpage,3,"Instagram, TikTok, YouTube"
94,geopop,3,"Instagram, TikTok, YouTube"
106,greenme,3,"Instagram, TikTok, YouTube"
155,internapoli,3,"Instagram, TikTok, YouTube"
216,milanotoday,3,"Instagram, TikTok, YouTube"
286,salernonotizie,3,"Instagram, TikTok, YouTube"
306,tgcom24,3,"Instagram, TikTok, YouTube"
329,vesuviolive,3,"Instagram, TikTok, YouTube"



Overlap using: source_identity_norm
Sources appearing on at least 2 platforms: 45
Sources appearing on all 3 platforms: 10


,source_identity_norm,n_platforms,platforms
33,cicap cicap,3,"Instagram, TikTok, YouTube"
74,fanpage fanpage,3,"Instagram, TikTok, YouTube"
101,geopop geopop,3,"Instagram, TikTok, YouTube"
113,greenme greenme,3,"Instagram, TikTok, YouTube"
172,internapoli internapoli,3,"Instagram, TikTok, YouTube"
246,milanotoday milanotoday,3,"Instagram, TikTok, YouTube"
322,salernonotizie salernonotizie,3,"Instagram, TikTok, YouTube"
371,vice vice,3,"Instagram, TikTok, YouTube"
375,vimeo vimeo,3,"Instagram, TikTok, YouTube"
391,youtube youtube,3,"Instagram, TikTok, YouTube"


In [74]:
# =============================================================================
# REMOVE GENERIC / PLATFORM-LIKE NORMALIZED SOURCES BEFORE MATCHING
# =============================================================================

MATCH_SOURCE_COL = "source_domain_norm"

GENERIC_SOURCE_NAMES = {
    "",
    "notizie",
    "news",
    "tv",
    "video",
    "online",
    "italia",
    "giornale",
    "quotidiano",
    "il",
    "la",
    "lo",
    "le",
    "gli",
    "youtube",
    "vimeo",
    "rumble"
}

all_videos_matching = all_videos[
    all_videos[MATCH_SOURCE_COL].notna() &
    (~all_videos[MATCH_SOURCE_COL].isin(GENERIC_SOURCE_NAMES))
].copy()

print("Matching source column:", MATCH_SOURCE_COL)
print("Videos retained for matching:", len(all_videos_matching))
print("Unique normalized sources retained:", all_videos_matching[MATCH_SOURCE_COL].nunique())

Matching source column: source_domain_norm
Videos retained for matching: 465359
Unique normalized sources retained: 355


# SOURCE OVERLAP BY PLATFORM PAIR

In [75]:
# =============================================================================
# SOURCE OVERLAP BY PLATFORM PAIR AFTER GENERIC-SOURCE FILTER
# =============================================================================

platform_pairs = [
    ("YouTube", "Instagram"),
    ("YouTube", "TikTok"),
    ("Instagram", "TikTok")
]

for p1, p2 in platform_pairs:

    sources_1 = set(
        all_videos_matching.loc[
            all_videos_matching["platform"] == p1,
            MATCH_SOURCE_COL
        ].dropna().unique()
    )

    sources_2 = set(
        all_videos_matching.loc[
            all_videos_matching["platform"] == p2,
            MATCH_SOURCE_COL
        ].dropna().unique()
    )

    overlap = sources_1.intersection(sources_2)

    print(f"{p1} - {p2}: {len(overlap)} overlapping normalized sources")
    print(sorted(list(overlap))[:30])
    print()

YouTube - Instagram: 23 overlapping normalized sources
['adnkronos', 'avvenire', 'cicap', 'fanpage', 'focus', 'gay', 'geopop', 'greenme', 'ilcapoluogo', 'ilmeteo', 'ilsicilia', 'internapoli', 'leggo', 'lercio', 'milanotoday', 'money', 'rainews', 'salernonotizie', 'strettoweb', 'telebari', 'umbria24', 'vice', 'wired']

YouTube - TikTok: 42 overlapping normalized sources
['100giornidaleoni', 'airc', 'avvenire', 'blogsicilia', 'cicap', 'cronachemaceratesi', 'eastwest', 'facta news', 'fanpage', 'focus', 'focusjunior', 'gay', 'geopop', 'greenme', 'grottaglieinrete', 'ilcapoluogo', 'ilmeteo', 'ilpost', 'ilsussidiario', 'iltirreno', 'insideover', 'internapoli', 'lagazzettadelmezzogiorno', 'lercio', 'lifegate', 'lindipendente online', 'medicitalia', 'milanocittastato', 'milanotoday', 'nanotv']

Instagram - TikTok: 30 overlapping normalized sources
['avvenire', 'cicap', 'fanpage', 'firenzetoday', 'focus', 'forbes italia', 'gay', 'geopop', 'greenme', 'huffpost', 'il giornale', 'ilcapoluogo', 'il

# GENERATE CANDIDATE CROSS-PLATFORM VIDEO MATCHES

In [76]:
from rapidfuzz import fuzz

DATE_WINDOW_DAYS = 2
TEXT_SIMILARITY_THRESHOLD = 85

platform_pairs = [
    ("YouTube", "Instagram"),
    ("YouTube", "TikTok"),
    ("Instagram", "TikTok")
]


def generate_platform_pair_candidates(
    df,
    platform_a,
    platform_b,
    source_col=MATCH_SOURCE_COL,
    date_window_days=DATE_WINDOW_DAYS
):
    """
    Generate candidate cross-platform video pairs.

    Candidate condition:
    - same normalized source/domain
    - different platform pair
    - publication dates within +/- date_window_days
    """

    a = df[df["platform"] == platform_a].copy()
    b = df[df["platform"] == platform_b].copy()

    a["date"] = pd.to_datetime(a["date"])
    b["date"] = pd.to_datetime(b["date"])

    expanded_a = []

    for offset in range(-date_window_days, date_window_days + 1):

        temp = a.copy()
        temp["candidate_date"] = temp["date"] + pd.Timedelta(days=offset)
        temp["date_offset"] = offset

        expanded_a.append(temp)

    expanded_a = pd.concat(expanded_a, ignore_index=True)

    candidates = expanded_a.merge(
        b,
        left_on=[source_col, "candidate_date"],
        right_on=[source_col, "date"],
        suffixes=("_a", "_b")
    )

    return candidates


def score_candidate_pairs(candidates):
    """
    Score candidate pairs using fuzzy text similarity and time difference.
    """

    candidates = candidates.copy()

    candidates["text_similarity"] = candidates.apply(
        lambda row: fuzz.token_set_ratio(
            row["match_text_clean_a"],
            row["match_text_clean_b"]
        ),
        axis=1
    )

    candidates["date_diff_days"] = (
        candidates["published_at_b"] - candidates["published_at_a"]
    ).dt.total_seconds().abs() / 86400

    return candidates

In [ ]:
# =============================================================================
# RUN CROSS-PLATFORM MATCHING
# =============================================================================

all_matches = []

for platform_a, platform_b in platform_pairs:

    print("\n" + "=" * 70)
    print(f"Matching {platform_a} vs {platform_b}")
    print("=" * 70)

    candidates = generate_platform_pair_candidates(
        all_videos_matching,
        platform_a,
        platform_b,
        source_col=MATCH_SOURCE_COL,
        date_window_days=DATE_WINDOW_DAYS
    )

    print("Candidate pairs before text scoring:", len(candidates))

    if len(candidates) == 0:
        continue

    candidates = score_candidate_pairs(candidates)

    matches = candidates[
        candidates["text_similarity"] >= TEXT_SIMILARITY_THRESHOLD
    ].copy()

    matches["platform_pair"] = platform_a + " - " + platform_b

    print("High-similarity matches:", len(matches))

    all_matches.append(matches)


cross_platform_matches = pd.concat(
    all_matches,
    ignore_index=True
)

print("\n" + "=" * 70)
print("FINAL MATCH SUMMARY")
print("=" * 70)

print("Total high-confidence candidate matches:", len(cross_platform_matches))
display(cross_platform_matches["platform_pair"].value_counts())



Matching YouTube vs Instagram
Candidate pairs before text scoring: 273583
High-similarity matches: 6571

Matching YouTube vs TikTok
Candidate pairs before text scoring: 228655
High-similarity matches: 5820

Matching Instagram vs TikTok
Candidate pairs before text scoring: 2283475
High-similarity matches: 38499

FINAL MATCH SUMMARY
Total high-confidence candidate matches: 50890


platform_pair
Instagram - TikTok     38499
YouTube - Instagram     6571
YouTube - TikTok        5820
Name: count, dtype: int64

,platform_pair,source_domain_norm,publisher_a,publisher_b,published_at_a,published_at_b,date_diff_days,text_similarity,views_a,views_b,url_a,url_b,match_text_clean_a,match_text_clean_b
25445,Instagram - TikTok,la stampa,La Stampa,la.stampa,2026-02-24 17:10:06+00:00,2026-02-24 17:10:14+00:00,0.000093,100.0,21317.0,107040.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@la.stampa/video/761048...,i magazzini coin all interno della stazione te...,i magazzini coin all interno della stazione te...
27892,Instagram - TikTok,fanpage,Fanpage.it,fanpage.it,2025-12-31 14:03:54+00:00,2025-12-31 13:09:24+00:00,0.037847,100.0,1347863.0,240581.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@fanpage.it/video/75900...,non vi preoccupate torno sono qua aveva detto ...,non vi preoccupate torno sono qua aveva detto ...
27885,Instagram - TikTok,la stampa,La Stampa,la.stampa,2025-12-31 15:30:02+00:00,2025-12-31 15:30:58+00:00,0.000648,100.0,94526.0,29895.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@la.stampa/video/759004...,prima della festa con i fuochi d artificio di ...,prima della festa con i fuochi d artificio di ...
27886,Instagram - TikTok,tgcom24,Tgcom24,tgcom24,2025-12-31 15:03:15+00:00,2025-12-31 15:21:25+00:00,0.012616,100.0,245644.0,708871.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@tgcom24/video/75900424...,in australia è già arrivato il sydney lo ha fe...,in australia è già arrivato il sydney lo ha fe...
27887,Instagram - TikTok,la repubblica,la Repubblica,la.repubblica,2025-12-31 15:00:27+00:00,2025-12-31 15:02:12+00:00,0.001215,100.0,1033910.0,173011.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@la.repubblica/video/75...,anthony hopkins celebra anni di sobrietà e lan...,anthony hopkins celebra anni di sobrietà e lan...
27888,Instagram - TikTok,la stampa,La Stampa,la.stampa,2025-12-31 15:00:16+00:00,2025-12-31 15:02:12+00:00,0.001343,100.0,196767.0,61990.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@la.stampa/video/759003...,anna danesi anni bresciana di roncadelle tre l...,anna danesi anni bresciana di roncadelle tre l...
27889,Instagram - TikTok,la stampa,La Stampa,la.stampa,2025-12-31 14:40:53+00:00,2025-12-31 14:40:45+00:00,0.000093,100.0,526209.0,12253.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@la.stampa/video/759003...,anthony hopkins celebra anni di sobrietà e lan...,anthony hopkins celebra anni di sobrietà e lan...
27890,Instagram - TikTok,rainews,RaiNews,rainews,2025-12-31 14:34:42+00:00,2025-12-31 14:37:50+00:00,0.002176,100.0,96736.0,24062.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@rainews/video/75900312...,durante la commemorazione la folla ha osservat...,durante la commemorazione la folla ha osservat...
27891,Instagram - TikTok,la repubblica,la Repubblica,la.repubblica,2025-12-31 14:30:03+00:00,2025-12-31 14:30:59+00:00,0.000648,100.0,377242.0,93559.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@la.repubblica/video/75...,prima della festa con i fuochi d artificio di ...,prima della festa con i fuochi d artificio di ...
27893,Instagram - TikTok,la repubblica,la Repubblica,la.repubblica,2025-12-31 13:54:29+00:00,2025-12-31 13:54:26+00:00,0.000035,100.0,3382694.0,1568921.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@la.repubblica/video/75...,letizia arnò è la ragazza in zaino e calzoncin...,letizia arnò è la ragazza in zaino e calzoncin...


In [78]:


display(
    cross_platform_matches[
        [
            "platform_pair",
            MATCH_SOURCE_COL,
            "publisher_a",
            "publisher_b",
            "published_at_a",
            "published_at_b",
            "date_diff_days",
            "text_similarity",
            "views_a",
            "views_b",
            "url_a",
            "url_b",
            "match_text_clean_a",
            "match_text_clean_b"
        ]
    ]
    .sort_values("text_similarity", ascending=True)
    .head(20)
)

,platform_pair,source_domain_norm,publisher_a,publisher_b,published_at_a,published_at_b,date_diff_days,text_similarity,views_a,views_b,url_a,url_b,match_text_clean_a,match_text_clean_b
9669,YouTube - TikTok,fanpage,Fanpage.it,fanpage.it,2023-01-24 15:30:09+00:00,2023-01-24 12:53:04+00:00,0.109086,85.000000,72113.0,7285.0,https://www.youtube.com/shorts/ng1KHZrkG-4,https://www.tiktok.com/@fanpage.it/video/71922...,evra si sfoga dopo la penalizzazione in serie ...,punti di penalizzazione dati alla juventus una...
2701,YouTube - Instagram,fanpage,Fanpage.it,Fanpage.it,2023-01-24 15:30:09+00:00,2023-01-24 12:51:20+00:00,0.110289,85.000000,72113.0,392173.0,https://www.youtube.com/shorts/ng1KHZrkG-4,https://lookaside.facebook.com/mcl/multimedia/...,evra si sfoga dopo la penalizzazione in serie ...,punti di penalizzazione dati alla juventus una...
2968,YouTube - Instagram,internapoli,InterNapoli,InterNapoli.it,2026-05-13 09:02:46+00:00,2026-05-13 11:43:02+00:00,0.111296,85.000000,62352.0,174673.0,https://www.youtube.com/shorts/-KzCPKTyoZc,https://lookaside.facebook.com/mcl/multimedia/...,tutta italia piange alessia la piccola guerrie...,anche gigi d alessio saluta la piccola alessia...
38,YouTube - Instagram,fanpage,Fanpage.it,Fanpage.it,2025-04-22 06:11:08+00:00,2025-04-20 11:23:05+00:00,1.783368,85.000000,13374.0,1107898.0,https://www.youtube.com/shorts/5SE10f8k5vM,https://lookaside.facebook.com/mcl/multimedia/...,piazza san pietro i fedeli cantano dopo il ros...,sorpresa di papa francesco ai fedeli oggi in p...
50671,Instagram - TikTok,fanpage,Fanpage.it,fanpage.it,2024-02-09 14:08:23+00:00,2024-02-11 08:00:00+00:00,1.744178,85.000000,188251.0,1417201.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@fanpage.it/video/73340...,i negramaro si raccontano nel nuovo format di ...,big mama si racconta nel nuovo format di fanpa...
9567,YouTube - TikTok,fanpage,Fanpage.it,fanpage.it,2024-06-12 15:30:21+00:00,2024-06-12 13:45:06+00:00,0.073090,85.000000,72867.0,25941.0,https://www.youtube.com/shorts/d8bfSU5OrsU,https://www.tiktok.com/@fanpage.it/video/73796...,la consigliera di fdi a lucca che ha chiesto u...,la consigliera di fdi a lucca che ha chiesto u...
1733,YouTube - Instagram,fanpage,Fanpage.it,Fanpage.it,2026-03-10 21:33:24+00:00,2026-03-10 21:32:19+00:00,0.000752,85.000000,25920.0,1288483.0,https://www.youtube.com/shorts/CFlXTQY8FsM,https://lookaside.facebook.com/mcl/multimedia/...,autobus in fiamme in svizzera ci sono diversi ...,immagini non adatte ad un pubblico sensibile t...
12501,Instagram - TikTok,ilmeteo,iLMeteo,ilmeteo,2023-07-14 11:32:36+00:00,2023-07-12 09:52:26+00:00,2.069560,85.000000,101697.0,187219.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@ilmeteo/video/72548677...,caldo record con caronte meteo ilmeteo meteo i...,super caldo con caronte
19208,Instagram - TikTok,fanpage,Fanpage.it,fanpage.it,2024-06-12 13:44:44+00:00,2024-06-12 13:45:06+00:00,0.000255,85.000000,992968.0,25941.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@fanpage.it/video/73796...,la consigliera di fdi a lucca che ha chiesto u...,la consigliera di fdi a lucca che ha chiesto u...
12123,YouTube - TikTok,fanpage,Fanpage.it,fanpage.it,2026-03-09 17:30:04+00:00,2026-03-10 18:22:43+00:00,1.036563,85.010707,16168.0,26009.0,https://www.youtube.com/shorts/DaEqupUD0Z0,https://www.tiktok.com/@fanpage.it/video/76156...,chi è mojtaba khamenei la nuova guida suprema ...,nel caso te lo fossi perso marzo l iran ha una...


In [80]:
# =============================================================================
# STRICTER MATCH REFINEMENT
# =============================================================================

from rapidfuzz import fuzz

refined_matches = cross_platform_matches.copy()

refined_matches["token_set_similarity"] = refined_matches["text_similarity"]

refined_matches["token_sort_similarity"] = refined_matches.apply(
    lambda row: fuzz.token_sort_ratio(
        row["match_text_clean_a"],
        row["match_text_clean_b"]
    ),
    axis=1
)

refined_matches["simple_similarity"] = refined_matches.apply(
    lambda row: fuzz.ratio(
        row["match_text_clean_a"],
        row["match_text_clean_b"]
    ),
    axis=1
)

refined_matches["text_len_a"] = refined_matches["match_text_clean_a"].str.len()
refined_matches["text_len_b"] = refined_matches["match_text_clean_b"].str.len()

refined_matches["min_text_len"] = refined_matches[
    ["text_len_a", "text_len_b"]
].min(axis=1)

refined_matches["max_text_len"] = refined_matches[
    ["text_len_a", "text_len_b"]
].max(axis=1)

refined_matches["text_length_ratio"] = (
    refined_matches["min_text_len"] /
    refined_matches["max_text_len"]
)

In [81]:
# =============================================================================
# DIAGNOSTIC: HOW MANY MATCHES SURVIVE DIFFERENT FILTERS?
# =============================================================================

diagnostic_rows = []

for token_set_threshold in [95, 98, 100]:
    for token_sort_threshold in [90, 95, 98]:
        for max_days in [0.25, 0.5, 1, 2]:

            temp = refined_matches[
                (refined_matches["token_set_similarity"] >= token_set_threshold) &
                (refined_matches["token_sort_similarity"] >= token_sort_threshold) &
                (refined_matches["date_diff_days"] <= max_days) &
                (refined_matches["min_text_len"] >= 50) &
                (refined_matches["text_length_ratio"] >= 0.5)
            ]

            diagnostic_rows.append({
                "token_set_threshold": token_set_threshold,
                "token_sort_threshold": token_sort_threshold,
                "max_days": max_days,
                "n_matches": len(temp),
                "youtube_instagram": (temp["platform_pair"] == "YouTube - Instagram").sum(),
                "youtube_tiktok": (temp["platform_pair"] == "YouTube - TikTok").sum(),
                "instagram_tiktok": (temp["platform_pair"] == "Instagram - TikTok").sum()
            })

diagnostic_summary = pd.DataFrame(diagnostic_rows)

display(
    diagnostic_summary
    .sort_values("n_matches", ascending=False)
)

,token_set_threshold,token_sort_threshold,max_days,n_matches,youtube_instagram,youtube_tiktok,instagram_tiktok
3,95,90,2.00,20806,2915,1229,16662
2,95,90,1.00,20573,2851,1185,16537
15,98,90,2.00,20424,2811,1184,16429
14,98,90,1.00,20211,2753,1144,16314
1,95,90,0.50,20123,2724,1098,16301
0,95,90,0.25,19918,2663,1085,16170
13,98,90,0.50,19793,2646,1059,16088
12,98,90,0.25,19595,2589,1047,15959
27,100,90,2.00,18455,1746,1042,15667
26,100,90,1.00,18291,1715,1009,15567


In [82]:
# =============================================================================
# FINAL STRICT HIGH-CONFIDENCE MATCHES
# =============================================================================

final_matches = refined_matches[
    (refined_matches["token_set_similarity"] >= 100) &
    (refined_matches["token_sort_similarity"] >= 98) &
    (refined_matches["simple_similarity"] >= 95) &
    (refined_matches["date_diff_days"] <= 0.25) &
    (refined_matches["min_text_len"] >= 50) &
    (refined_matches["text_length_ratio"] >= 0.8)
].copy()

print("Loose candidate matches:", len(cross_platform_matches))
print("Strict high-confidence matches:", len(final_matches))

display(
    final_matches["platform_pair"]
    .value_counts()
    .rename_axis("platform_pair")
    .reset_index(name="n_matches")
)

Loose candidate matches: 50890
Strict high-confidence matches: 12071


,platform_pair,n_matches
0,Instagram - TikTok,11563
1,YouTube - Instagram,268
2,YouTube - TikTok,240


In [83]:
display(
    final_matches[
        [
            "platform_pair",
            MATCH_SOURCE_COL,
            "publisher_a",
            "publisher_b",
            "published_at_a",
            "published_at_b",
            "date_diff_days",
            "token_set_similarity",
            "token_sort_similarity",
            "simple_similarity",
            "text_length_ratio",
            "views_a",
            "views_b",
            "url_a",
            "url_b",
            "match_text_clean_a",
            "match_text_clean_b"
        ]
    ]
    .sort_values(
        ["platform_pair", "simple_similarity", "token_sort_similarity"],
        ascending=[True, False, False]
    )
    .head(30)
)

,platform_pair,source_domain_norm,publisher_a,publisher_b,published_at_a,published_at_b,date_diff_days,token_set_similarity,token_sort_similarity,simple_similarity,text_length_ratio,views_a,views_b,url_a,url_b,match_text_clean_a,match_text_clean_b
12915,Instagram - TikTok,vice,VICE,vice,2025-10-08 00:39:34+00:00,2025-10-07 23:39:06+00:00,0.041991,100.0,100.0,100.0,1.0,171070.0,9832.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@vice/video/75586283947...,tame impala would love to make a hip hop album...,tame impala would love to make a hip hop album...
12921,Instagram - TikTok,wikipedia,Wikipedia,wikipedia,2025-09-28 00:57:53+00:00,2025-09-27 20:10:20+00:00,0.199687,100.0,100.0,100.0,1.0,24044.0,13648.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@wikipedia/video/755486...,a barrel roll is a classic flying maneuver whe...,a barrel roll is a classic flying maneuver whe...
12953,Instagram - TikTok,vice,VICE,vice,2025-07-21 00:01:06+00:00,2025-07-20 23:59:07+00:00,0.001377,100.0,100.0,100.0,1.0,2115833.0,599180.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@vice/video/75293178502...,ozzy osbourne recalls the odd encounter he had...,ozzy osbourne recalls the odd encounter he had...
13051,Instagram - TikTok,la repubblica,la Repubblica,la.repubblica,2025-02-12 00:00:02+00:00,2025-02-11 23:58:37+00:00,0.000984,100.0,100.0,100.0,1.0,1741452.0,156105.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@la.repubblica/video/74...,una scatolina rosa con su scritto il titolo de...,una scatolina rosa con su scritto il titolo de...
13142,Instagram - TikTok,la repubblica,la Repubblica,la.repubblica,2023-03-23 15:00:00+00:00,2023-03-23 14:33:24+00:00,0.018472,100.0,100.0,100.0,1.0,892106.0,5149.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@la.repubblica/video/72...,c è questa retorica americana del fai quello c...,c è questa retorica americana del fai quello c...
13143,Instagram - TikTok,internapoli,InterNapoli.it,internapoli.it,2023-03-23 13:42:03+00:00,2023-03-23 13:45:46+00:00,0.002581,100.0,100.0,100.0,1.0,145548.0,44143.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@internapoli.it/video/7...,il capo della squadra mobile di napoli alfredo...,il capo della squadra mobile di napoli alfredo...
13156,Instagram - TikTok,la repubblica,la Repubblica,la.repubblica,2023-03-21 18:00:00+00:00,2023-03-21 15:59:41+00:00,0.083553,100.0,100.0,100.0,1.0,2047276.0,4602144.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@la.repubblica/video/72...,sono stato dipendente dalla cocaina per qualch...,sono stato dipendente dalla cocaina per qualch...
13193,Instagram - TikTok,fanpage,Fanpage.it,fanpage.it,2023-03-16 12:04:49+00:00,2023-03-16 12:17:40+00:00,0.008924,100.0,100.0,100.0,1.0,1781564.0,41192.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@fanpage.it/video/72111...,il momento dello scontro tra il drone usa e il...,il momento dello scontro tra il drone usa e il...
13207,Instagram - TikTok,internapoli,InterNapoli.it,internapoli.it,2023-03-15 11:44:03+00:00,2023-03-15 11:46:06+00:00,0.001424,100.0,100.0,100.0,1.0,139233.0,3230.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@internapoli.it/video/7...,hotel continental royal blindato per la presen...,hotel continental royal blindato per la presen...
13225,Instagram - TikTok,fanpage,Fanpage.it,fanpage.it,2023-03-12 15:27:42+00:00,2023-03-12 15:30:08+00:00,0.001690,100.0,100.0,100.0,1.0,1148424.0,10107.0,https://lookaside.facebook.com/mcl/multimedia/...,https://www.tiktok.com/@fanpage.it/video/72096...,la sfilata di versace arriva al termine della ...,la sfilata di versace arriva al termine della ...


In [ ]:
# =============================================================================
# KEEP BEST MATCH PER VIDEO ON EACH PLATFORM PAIR IN CASE OF DUPLICATES
# =============================================================================

final_matches = final_matches.sort_values(
    [
        "platform_pair",
        "video_id_a",
        "simple_similarity",
        "token_sort_similarity",
        "date_diff_days"
    ],
    ascending=[True, True, False, False, True]
)

best_matches = (
    final_matches
    .drop_duplicates(subset=["platform_pair", "video_id_a"], keep="first")
    .copy()
)

best_matches = best_matches.sort_values(
    [
        "platform_pair",
        "video_id_b",
        "simple_similarity",
        "token_sort_similarity",
        "date_diff_days"
    ],
    ascending=[True, True, False, False, True]
)

best_matches = (
    best_matches
    .drop_duplicates(subset=["platform_pair", "video_id_b"], keep="first")
    .copy()
)

print("Strict matches before one-to-one filtering:", len(final_matches))
print("Best one-to-one matches:", len(best_matches))

display(
    best_matches["platform_pair"]
    .value_counts()
    .rename_axis("platform_pair")
    .reset_index(name="n_matches")
)

Strict matches before one-to-one filtering: 12071
Best one-to-one matches: 12044


,platform_pair,n_matches
0,Instagram - TikTok,11537
1,YouTube - Instagram,268
2,YouTube - TikTok,239
